# 🔍 Model Explainability Notebook
## Feature Importance · SHAP Values · Permutation Importance

Understanding why the model predicts churn for each customer.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
print("Ready ✓")


## 1. Load Data & Model

In [ ]:
from src.data.pipeline import ChurnDataPipeline, PipelineConfig
import tensorflow as tf

pipeline = ChurnDataPipeline(PipelineConfig(save_artifacts=False, artifacts_dir='/tmp/expl'))
split = pipeline.run('../data/raw/ChurnPrediction.csv')

FEATURE_NAMES = [
    'CreditScore','Geography','Gender','Age','Tenure',
    'Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary'
]

try:
    import glob
    ckpts = glob.glob('../artifacts/checkpoints/best_model.keras')
    model = tf.keras.models.load_model(ckpts[0])
    print(f"Loaded: {ckpts[0]}")
except:
    from src.models.ann import ChurnANN, baseline_config
    from src.training.trainer import ChurnModelTrainer, TrainingConfig
    model = ChurnANN(baseline_config(10)).build()
    t = ChurnModelTrainer(TrainingConfig(epochs=30, batch_size=32, verbose=0,
                                         checkpoint_dir='/tmp/expl/ckpt'))
    t.train(model, split.X_train, split.y_train, split.X_val, split.y_val)
    print("Trained fresh model")


## 2. Permutation Feature Importance

In [ ]:
from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(split.y_test, model.predict(split.X_test, verbose=0))
print(f"Baseline AUC: {baseline_auc:.4f}\n")

importances = {}
for i, feat in enumerate(FEATURE_NAMES):
    X_perm = split.X_test.copy()
    np.random.shuffle(X_perm[:, i])
    perm_auc = roc_auc_score(split.y_test, model.predict(X_perm, verbose=0))
    importances[feat] = baseline_auc - perm_auc

sorted_imp = dict(sorted(importances.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in sorted_imp.values()]
ax.barh(list(sorted_imp.keys())[::-1], list(sorted_imp.values())[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('AUC Drop (higher = more important)')
ax.set_title('Permutation Feature Importance', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()
print("Top 3 features:", list(sorted_imp.keys())[:3])


## 3. SHAP Values (if shap installed)

In [ ]:
try:
    import shap
    shap.initjs()

    explainer = shap.KernelExplainer(
        lambda x: model.predict(x, verbose=0),
        shap.sample(split.X_train, 100),
    )
    shap_values = explainer.shap_values(split.X_test[:200], nsamples=100)

    fig, ax = plt.subplots(figsize=(9, 5))
    shap.summary_plot(shap_values, split.X_test[:200],
                      feature_names=FEATURE_NAMES,
                      show=False)
    plt.title('SHAP Summary Plot', fontsize=13)
    plt.tight_layout(); plt.show()
    print("SHAP analysis complete ✓")

except ImportError:
    print("shap not installed. Run: pip install shap")
    print("Showing permutation importance as proxy.")


## 4. Individual Prediction Explanation

In [ ]:
# Manual sensitivity analysis for a single customer
sample_idx = 5
x_sample = split.X_test[sample_idx:sample_idx+1]
base_prob = float(model.predict(x_sample, verbose=0)[0][0])
print(f"Customer #{sample_idx} — Churn probability: {base_prob:.3f}")
print(f"Prediction: {'CHURN' if base_prob >= 0.5 else 'RETAIN'}\n")

# Perturb each feature ± 1 std to show sensitivity
sensitivities = {}
for i, feat in enumerate(FEATURE_NAMES):
    x_hi = x_sample.copy(); x_hi[0, i] += 1.0
    x_lo = x_sample.copy(); x_lo[0, i] -= 1.0
    p_hi = float(model.predict(x_hi, verbose=0)[0][0])
    p_lo = float(model.predict(x_lo, verbose=0)[0][0])
    sensitivities[feat] = (p_hi - p_lo) / 2

sorted_sens = dict(sorted(sensitivities.items(), key=lambda x: abs(x[1]), reverse=True))
print("Feature sensitivity (Δ probability per +1 std):")
for f, s in sorted_sens.items():
    bar = '▲' if s > 0 else '▼'
    print(f"  {f:<20} {bar} {abs(s):.4f}")
